Lectrue 1 文本的切分和向量化

In [1]:
import os
# 核心魔法：把 Hugging Face 的下载请求全部重定向到镜像站
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
"""
下载一个名为 all-mpnet-base-v2 的分词器（Tokenizer）模型文件。
"""
import textwrap
import chromadb
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter, SentenceTransformersTokenTextSplitter
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

D:\dev\py3.14\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def word_wrap(text, width=80):
    """自动给长文本换行，让打印出来的结果更易读"""
    return textwrap.fill(text, width=width)

In [3]:
# ==========================================
# 1. 加载并清洗 PDF 数据
# ==========================================
print("正在加载 PDF 文件...")
reader = PdfReader("microsoft_annual_report_2022.pdf")
pdf_texts = [p.extract_text().strip() for p in reader.pages]
pdf_texts = [text for text in pdf_texts if text] # 过滤空页
print("PDF 加载完成！\n")

正在加载 PDF 文件...
PDF 加载完成！



In [4]:
print("正在进行文本切分...")
# 第一步：按字符和段落切分 (最大1000字符)
character_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=1000,
    chunk_overlap=0
)
character_split_texts = character_splitter.split_text('\n\n'.join(pdf_texts))

正在进行文本切分...


In [ ]:
# 第二步：按 Token 切分 (严格限制最大256 Token)
token_splitter = SentenceTransformersTokenTextSplitter(
    chunk_overlap=0,
    tokens_per_chunk=256
)

token_split_texts = []
for text in character_split_texts:
    # 注意这里的 +=，它会把所有小块平铺到一个大列表里
    token_split_texts += token_splitter.split_text(text)

print(f"切分完毕！共生成 {len(token_split_texts)} 个纯净的 Token 文本块。\n")



In [9]:
print("正在启动 Chroma 数据库并进行向量化 (这步可能需要下载模型或花几秒钟计算)...")
embedding_function = SentenceTransformerEmbeddingFunction()

# 启动内存模式的 Chroma
chroma_client = chromadb.Client()

# 如果之前跑过报错了，可以先清空同名集合避免冲突
if "microsoft_annual_report_2022" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("microsoft_annual_report_2022")

chroma_collection = chroma_client.create_collection(
    name="microsoft_annual_report_2022",
    embedding_function=embedding_function
)

# 生成唯一 ID
ids = [str(i) for i in range(len(token_split_texts))]

# 批量存入数据库 (纯文本进去，底层自动转成 384 维向量)
chroma_collection.add(ids=ids, documents=token_split_texts)
print(f"数据入库成功！当前数据库包含 {chroma_collection.count()} 条记录。\n")

正在启动 Chroma 数据库并进行向量化 (这步可能需要下载模型或花几秒钟计算)...
数据入库成功！当前数据库包含 349 条记录。



In [10]:
query = "What was the total revenue?"
print(f"【用户提问】: {query}\n")

# 检索最相关的 5 个结果
results = chroma_collection.query(
    query_texts=[query],
    n_results=5
)


【用户提问】: What was the total revenue?



In [12]:
retrieved_documents = results['documents'][0]
distances = results['distances'][0]

In [13]:
print("【检索结果 (Top 5)】:")
for i, (document, distance) in enumerate(zip(retrieved_documents, distances)):
    print(f"--- 结果 {i+1} (距离得分: {distance:.4f}) ---")
    print(word_wrap(document))
    print("\n")

【检索结果 (Top 5)】:
--- 结果 1 (距离得分: 0.4293) ---
74 note 13 — unearned revenue unearned revenue by segment was as follows : ( in
millions ) june 30, 2022 2021 productivity and business processes $ 24, 558 $
22, 120 intelligent cloud 19, 371 17, 710 more personal computing 4, 479 4, 311
total $ 48, 408 $ 44, 141 changes in unearned revenue were as follows : ( in
millions ) year ended june 30, 2022 balance, beginning of period $ 44, 141
deferral of revenue 110, 455 recognition of unearned revenue ( 106, 188 )
balance, end of period $ 48, 408 revenue allocated to remaining performance
obligations, which includes unearned revenue and amounts that will be invoiced
and recognized as revenue in future periods, was $ 193 billion as of june 30,
2022, of which $ 189 billion is related to the commercial portion of revenue. we
expect to recognize approximately 45 % of this revenue over the next 12 months
and the remainder thereafter. note 14 — leases


--- 结果 2 (距离得分: 0.4562) ---
that are not sold sepa